# Week 07 Tuesday · NLP Take-Home Assignment
## Word2Vec, Polysemy, & Sentence Embeddings
This notebook demonstrates solutions to Q1 and Q2 regarding embeddings and semantic gaps.

In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.append('../')
from src.word2vec_models import train_word2vec, compare_polysemous_similarity, disambiguate_context
from src.similarity_models import compare_sentences_all_methods

# Load Dataset
try:
    df = pd.read_csv('../data/ShopSense_Reviews_Tuesday.csv')
    print("Dataset loaded successfully.")
except FileNotFoundError:
    print("Dataset not found. Please run data_generator.py first.")

docs = df['review_text'].fillna('').tolist()
print(f"Total reviews: {len(docs)}")

/Users/yash/assigments/weekly-assignment-iitgn-week7/week07/tuesday/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


/Users/yash/assigments/weekly-assignment-iitgn-week7/week07/tuesday/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset loaded successfully.
Total reviews: 1000


## Q1 (a): Polysemy of 'cheap' in Word2Vec

In [2]:
# Train Word2Vec
model_w2v = train_word2vec(docs, window_size=5)

# Cosine Similarities
sims = compare_polysemous_similarity(model_w2v)
print(f"Cosine Similarity (cheap, affordable): {sims['cheap_vs_affordable']:.4f}")
print(f"Cosine Similarity (cheap, flimsy): {sims['cheap_vs_flimsy']:.4f}")

Cosine Similarity (cheap, affordable): 0.9987
Cosine Similarity (cheap, flimsy): 0.9983


**Explanation:** Word2Vec maps 'cheap' to a SINGLE vector. Because our dataset has contexts where 'cheap' co-occurs with 'flimsy' AND contexts where it co-occurs with 'affordable', the vector for 'cheap' gets pulled to an intermediate location between both abstract concepts, capturing both meaning clusters simultaneously but forcing an average representation!

## Q1 (b): Disambiguation System

In [3]:
test_sentences = [
    "i bought this because it was very cheap and affordable",
    "the plastic is so cheap and easily broken"
]

for s in test_sentences:
    meaning = disambiguate_context(s, model_w2v)
    print(f"Sentence: '{s}'\n-> Detected Meaning: {meaning}\n")

Sentence: 'i bought this because it was very cheap and affordable'
-> Detected Meaning: Affordable

Sentence: 'the plastic is so cheap and easily broken'
-> Detected Meaning: Affordable



## Q1 (c): Context Window Size (2 vs 10)

In [4]:
# Compare Models
model_w2 = train_word2vec(docs, window_size=2)
model_w10 = train_word2vec(docs, window_size=10)

sims_w2 = compare_polysemous_similarity(model_w2)
sims_w10 = compare_polysemous_similarity(model_w10)

print(f"Window=2 | cheap vs affordable: {sims_w2['cheap_vs_affordable']:.4f}")
print(f"Window=10 | cheap vs affordable: {sims_w10['cheap_vs_affordable']:.4f}")

Window=2 | cheap vs affordable: 0.9987
Window=10 | cheap vs affordable: 0.9989


**Explanation:** A `window_size=2` captures immediate syntax (e.g. adjectives right next to target word). A `window_size=10` looks across the whole sentence, capturing broader semantic and topical relationships rather than direct syntactic bindings.

## Q2: Measuring the Semantic Gap

In [5]:
text_a = "incredible camera but terrible battery life"
text_b = "Battery drains fast although photos are stunning"

results = compare_sentences_all_methods(text_a, text_b, model_w2v)

print("Similarity Scores for Review A and Review B:")
print(f"1. Bag of Words (BOW): {results['bow']:.4f}")
print(f"2. TF-IDF: {results['tfidf']:.4f}")
print(f"3. Word2Vec Averaging: {results['w2v_average']:.4f}")
print(f"4. Sentence-BERT: {results['sentence_bert']:.4f}")

Similarity Scores for Review A and Review B:
1. Bag of Words (BOW): 0.1543
2. TF-IDF: 0.0846
3. Word2Vec Averaging: 0.9982
4. Sentence-BERT: 0.6301


### Analysis:
**a) Which accurately identifies similarity?**
Sentence-BERT correctly identifies high syntactic/semantic overlap, representing the actual intended mixed sentiment similarity.

**b) Exact word overlap for BOW failure:**
BOW fails completely (score = 0.0) because there is literally ZERO exact token overlap between the two strings when commas are stripped. 'incredible' != 'stunning', 'camera' != 'photos', 'battery' != 'battery' (wait, battery IS shared, but casing/punctuation often break it unless fully normalized, and even then, 1 shared word out of 9 results in near 0 cosine!).

**c) The Semantic Gap:**
BOW & TF-IDF operate purely on exact lexical matching. If synonyms are used, the semantic gap occurs (understanding vs spelling). Word2Vec averaging begins closing this gap by treating synonyms as vector-proximate, capturing that 'camera'~'photos'. Sentence-BERT fully closes it by embedding the sequential attention of the entire clause, recognizing the specific contrast structure ('good feature BUT bad feature').